# Setup

In [1]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd
import numpy as np

In [3]:
PROJECT_ID = "river-blade-112921" #change project ID here

ED = "physionet-data.mimiciv_ed"
HOSP = "physionet-data.mimiciv_2_2_hosp"
ICU = "physionet-data.mimiciv_2_2_icu"

client = bigquery.Client(project=PROJECT_ID)

print("Connected to project:", PROJECT_ID)
print("ED dataset:", ED)
print("Hospital dataset:", HOSP)
print("ICU dataset:", ICU)

Connected to project: river-blade-112921
ED dataset: physionet-data.mimiciv_ed
Hospital dataset: physionet-data.mimiciv_2_2_hosp
ICU dataset: physionet-data.mimiciv_2_2_icu


# Data extraction and exploration

In [10]:
#Get ED encounters
query = f"""
SELECT *
FROM `{ED}.edstays`
"""

edstays_raw = client.query(query).result().to_dataframe(
    create_bqstorage_client=False
)

# Check the records
print("ED encounters:", len(edstays_raw))
print("Unique ED stays:", edstays_raw["stay_id"].nunique())
print("Unique patients:", edstays_raw["subject_id"].nunique())
print("Duplicate rows:", edstays_raw.duplicated().sum())

# Count encounters linked to hospital admission
print("Admitted encounters:", edstays_raw["hadm_id"].notna().sum())

# Check missing values
print("\nMissing values:")
print(edstays_raw.isna().sum())

ED encounters: 425087
Unique ED stays: 425087
Unique patients: 205504
Duplicate rows: 0
Admitted encounters: 203016

Missing values:
subject_id                0
hadm_id              222071
stay_id                   0
intime                    0
outtime                   0
gender                    0
race                      0
arrival_transport         0
disposition               0
dtype: int64


In [12]:
# Get patient IDs from ED encounters
ed_patient_ids = edstays_raw["subject_id"].dropna().unique().tolist()

#Get demographics for these patients
query = f"""
SELECT *
FROM `{HOSP}.patients`
WHERE subject_id IN UNNEST(@patient_ids)
"""

job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ArrayQueryParameter(
            "patient_ids", "INT64", ed_patient_ids
        )
    ]
)

patients_raw = client.query(
    query, job_config=job_config
).result().to_dataframe(create_bqstorage_client=False)

# Check the records
print("Patient records:", len(patients_raw))
print("Unique patients:", patients_raw["subject_id"].nunique())
print("Duplicate rows:", patients_raw.duplicated().sum())

# Check whether any ED patients are missing
missing_patients = ~pd.Series(ed_patient_ids).isin(patients_raw["subject_id"])
print("ED patients without demographics:", missing_patients.sum())

Patient records: 205504
Unique patients: 205504
Duplicate rows: 0
ED patients without demographics: 0


In [13]:
# Get patient IDs from ED encounters
ed_patient_ids = edstays_raw["subject_id"].dropna().unique().tolist()

#Get admissions for these patients
query = f"""
SELECT *
FROM `{HOSP}.admissions`
WHERE subject_id IN UNNEST(@patient_ids)
"""

job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ArrayQueryParameter(
            "patient_ids", "INT64", ed_patient_ids
        )
    ]
)

admissions_raw = client.query(
    query, job_config=job_config
).result().to_dataframe(create_bqstorage_client=False)

# Check the records
print("Hospital admissions:", len(admissions_raw))
print("Unique admissions:", admissions_raw["hadm_id"].nunique())
print("Unique patients:", admissions_raw["subject_id"].nunique())
print("Duplicate rows:", admissions_raw.duplicated().sum())

# Check whether ED admission IDs exist in the hospital table
ed_admission_ids = edstays_raw["hadm_id"].dropna()
missing_admissions = ~ed_admission_ids.isin(admissions_raw["hadm_id"])

print("ED encounters without a matching admission:", missing_admissions.sum())

# Check missing values
print("\nMissing values:")
print(admissions_raw.isna().sum())

Hospital admissions: 332101
Unique admissions: 332101
Unique patients: 122780
Duplicate rows: 0
ED encounters without a matching admission: 0

Missing values:
subject_id                   0
hadm_id                      0
admittime                    0
dischtime                    0
deathtime               326919
admission_type               0
admit_provider_id            4
admission_location           0
discharge_location      101616
insurance                    0
language                     0
marital_status            5690
race                         0
edregtime                85687
edouttime                85687
hospital_expire_flag         0
dtype: int64


In [9]:
#Get triage information
query = f"""
SELECT *
FROM `{ED}.triage`
"""

triage_raw = client.query(query).result().to_dataframe(
    create_bqstorage_client=False
)

# Check the number of records
print("Triage records:", len(triage_raw))
print("Unique ED stays:", triage_raw["stay_id"].nunique())
print("Duplicate rows:", triage_raw.duplicated().sum())

# Check how many ED stays have no triage record
missing_triage = ~edstays_raw["stay_id"].isin(triage_raw["stay_id"])
print("ED stays without triage:", missing_triage.sum())

# Check missing values in each column
print("\nMissing values:")
print(triage_raw.isna().sum())

Triage records: 425087
Unique ED stays: 425087
Duplicate rows: 0
ED stays without triage: 0

Missing values:
subject_id            0
stay_id               0
temperature       23415
heartrate         17090
resprate          20353
o2sat             20596
sbp               18291
dbp               19091
pain              12896
acuity             6987
chiefcomplaint       22
dtype: int64


In [14]:
# Summarise the six vital signs
vitals = ["temperature", "heartrate", "resprate", "o2sat", "sbp", "dbp"]

display(triage_raw[vitals].describe())

# Count encounters missing all six vital signs
all_vitals_missing = triage_raw[vitals].isna().all(axis=1)
print("All six vital signs missing:", all_vitals_missing.sum())

# Check the recorded acuity categories
print("\nAcuity counts:")
print(triage_raw["acuity"].value_counts(dropna=False).sort_index())

# Inspect common pain entries
print("\nMost common pain entries:")
print(triage_raw["pain"].value_counts(dropna=False).head(20))

# Count chief complaints that contain only spaces or are empty
blank_complaints = triage_raw["chiefcomplaint"].str.strip().eq("")
print("\nBlank chief complaints:", blank_complaints.sum())

,temperature,heartrate,resprate,o2sat,sbp,dbp
count,401672,407997,404734,404491,406796,405996
unique,517,220,83,107,274,527
top,98.000000000,80.000000000,18.000000000,100.000000000,130.000000000,80.000000000
freq,30622,16886,158810,170428,9091,13335


All six vital signs missing: 15793

Acuity counts:
acuity
1.000000000     24019
2.000000000    139411
3.000000000    225066
4.000000000     28504
5.000000000      1100
None             6987
Name: count, dtype: int64

Most common pain entries:
pain
0           140719
8            41730
10           40914
7            31423
5            29210
6            25206
9            20210
4            19721
3            16115
2            13903
None         12896
13           10921
1             5903
unable        3606
Critical      1697
uta           1601
UTA           1577
ua            1201
UA             478
u/a            356
Name: count, dtype: int64

Blank chief complaints: 0


In [15]:
# Create a working copy of triage
triage = triage_raw.copy()

# Convert vital signs to numbers
vitals = ["temperature", "heartrate", "resprate", "o2sat", "sbp", "dbp"]

for column in vitals:
    triage[column] = pd.to_numeric(triage[column], errors="coerce")

# Check whether conversion created any extra missing values
print("Extra missing values after conversion:")
print(triage[vitals].isna().sum() - triage_raw[vitals].isna().sum())

# Show numeric summaries
display(triage[vitals].describe())

Extra missing values after conversion:
temperature    0
heartrate      0
resprate       0
o2sat          0
sbp            0
dbp            0
dtype: int64


,temperature,heartrate,resprate,o2sat,sbp,dbp
count,401672.000000,407997.000000,404734.000000,404491.000000,406796.000000,405996.000000
mean,98.015046,85.079891,17.565521,98.471888,135.395352,81.262126
std,4.008575,18.041690,5.485706,17.040807,240.956408,1057.220031
min,0.100000,1.000000,0.000000,0.000000,1.000000,0.000000
25%,97.500000,72.000000,16.000000,97.000000,120.000000,68.000000
50%,98.000000,84.000000,18.000000,99.000000,133.000000,77.000000
75%,98.600000,96.000000,18.000000,100.000000,148.000000,87.000000
max,986.000000,1228.000000,1820.000000,9322.000000,151103.000000,661672.000000


In [16]:
# Inspect the lowest and highest values for each vital sign
for column in vitals:
    print("\n", column)

    print("Lowest 10 distinct values:")
    print(triage[column].dropna().sort_values().unique()[:10])

    print("Highest 10 distinct values:")
    print(triage[column].dropna().sort_values().unique()[-10:])


 temperature
Lowest 10 distinct values:
[0.1  0.3  0.5  0.98 1.   1.04 2.   5.   6.   6.2 ]
Highest 10 distinct values:
[111.4  123.85 138.92 146.88 148.74 977.   978.   979.   981.   986.  ]

 heartrate
Lowest 10 distinct values:
[ 1.  7.  8.  9. 10. 11. 12. 13. 14. 16.]
Highest 10 distinct values:
[ 234.  250.  256.  570.  796.  922.  978. 1007. 1109. 1228.]

 resprate
Lowest 10 distinct values:
[ 0.  1.  2.  4.  6.  7.  8.  9. 10. 11.]
Highest 10 distinct values:
[ 161.   162.   165.   187.   189.   198.   209.  1796.5 1797.9 1820. ]

 o2sat
Lowest 10 distinct values:
[ 0.  2.  4.  5.  6.  7.  8.  9. 10. 15.]
Highest 10 distinct values:
[1001. 1002. 1003. 1004. 1005. 1007. 1009. 1999. 2100. 9322.]

 sbp
Lowest 10 distinct values:
[ 1.  5.  7.  8.  9. 10. 11. 12. 13. 14.]
Highest 10 distinct values:
[   776.    935.    941.    946.    948.    960.   9656.  12080.  19734.
 151103.]

 dbp
Lowest 10 distinct values:
[0. 1. 2. 3. 4. 5. 6. 7. 8. 9.]
Highest 10 distinct values:
[  9115.  

# Cohort Construction

Add demographics to ED encounters

In [17]:
# Keep ED encounters linked to a hospital admission
cohort = edstays_raw[edstays_raw["hadm_id"].notna()].copy()

print("Admitted ED encounters:", len(cohort))

# Select the demographic fields we need
demographics = patients_raw[
    ["subject_id", "anchor_age", "anchor_year", "anchor_year_group"]
]

# Add demographics to each encounter
cohort = cohort.merge(
    demographics,
    on="subject_id",
    how="left",
    validate="many_to_one"
)

# Calculate age at ED arrival
cohort["intime"] = pd.to_datetime(cohort["intime"])

cohort["age"] = (
    cohort["anchor_age"] + cohort["intime"].dt.year - cohort["anchor_year"]
)

# Inspect age before applying the adult restriction
print("Encounters after merge:", len(cohort))
print("Unique ED stays:", cohort["stay_id"].nunique())
print("Encounters aged under 18:", (cohort["age"] < 18).sum())

display(cohort["age"].describe())

Admitted ED encounters: 203016
Encounters after merge: 203016
Unique ED stays: 203016
Encounters aged under 18: 0


,age
count,203016.0
mean,59.985119
std,19.54087
min,18.0
25%,47.0
50%,61.0
75%,75.0
max,103.0


Adding hospital times asnd check timing

In [18]:
# Keep only adults (It wont remove anyone)
cohort = cohort[cohort["age"] >= 18].copy()

# Select hospital admission and discharge times
hospital_times = admissions_raw[
    ["subject_id", "hadm_id", "admittime", "dischtime"]
]

# Add hospital times
cohort = cohort.merge(
    hospital_times,
    on=["subject_id", "hadm_id"],
    how="left",
    validate="many_to_one"
)

# Convert time columns to datetime
time_columns = ["intime", "outtime", "admittime", "dischtime"]

for column in time_columns:
    cohort[column] = pd.to_datetime(cohort[column])

# Check missing times
print("Missing times:")
print(cohort[time_columns].isna().sum())

# Check timing problems
bad_ed_times = cohort["outtime"] < cohort["intime"]
bad_discharge_times = cohort["dischtime"] <= cohort["intime"]

print("\nED departure before arrival:", bad_ed_times.sum())
print("Hospital discharge at or before ED arrival:", bad_discharge_times.sum())

print("\nEncounters after merge:", len(cohort))
print("Unique ED stays:", cohort["stay_id"].nunique())

Missing times:
intime       0
outtime      0
admittime    0
dischtime    0
dtype: int64

ED departure before arrival: 0
Hospital discharge at or before ED arrival: 42

Encounters after merge: 203016
Unique ED stays: 203016


Remove invlaid timing and add triage information

In [19]:
# Keep only encounters with valid timing
valid_times = (
    (cohort["outtime"] >= cohort["intime"])
    & (cohort["dischtime"] > cohort["intime"])
)

print("Excluded for invalid timing:", (~valid_times).sum())

cohort = cohort[valid_times].copy()

# Add the triage information converted to numeric earlier
cohort = cohort.merge(
    triage,
    on=["subject_id", "stay_id"],
    how="left",
    validate="one_to_one"
)

# Check the cohort size
print("Remaining encounters:", len(cohort))
print("Unique ED stays:", cohort["stay_id"].nunique())
print("Unique patients:", cohort["subject_id"].nunique())

Excluded for invalid timing: 42
Remaining encounters: 202974
Unique ED stays: 202974
Unique patients: 107367


Diagnosis records for corhot patients. Including all their hospital admissions to identify earlier documented ESKD or chornic dialysis

In [20]:
# Get the patients in our cohort
cohort_patient_ids = cohort["subject_id"].unique().tolist()

#Get their hospital diagnosis records
query = f"""
SELECT subject_id, hadm_id, seq_num, icd_code, icd_version
FROM `{HOSP}.diagnoses_icd`
WHERE subject_id IN UNNEST(@patient_ids)
"""

job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ArrayQueryParameter(
            "patient_ids", "INT64", cohort_patient_ids
        )
    ]
)

diagnoses_raw = client.query(
    query, job_config=job_config
).result().to_dataframe(create_bqstorage_client=False)

# Check the downloaded records
print("Diagnosis records:", len(diagnoses_raw))
print("Unique patients:", diagnoses_raw["subject_id"].nunique())
print("Unique admissions:", diagnoses_raw["hadm_id"].nunique())
print("Duplicate rows:", diagnoses_raw.duplicated().sum())

print("\nICD version counts:")
print(diagnoses_raw["icd_version"].value_counts())

Diagnosis records: 3617741
Unique patients: 107345
Unique admissions: 306657
Duplicate rows: 0

ICD version counts:
icd_version
9     1845440
10    1772301
Name: count, dtype: Int64


Renal diagnosis codes

In [21]:
# Create a working copy
diagnoses = diagnoses_raw.copy()

# Standardise the code text
diagnoses["icd_code"] = diagnoses["icd_code"].str.strip().str.upper()

# Flag ESKD codes
diagnoses["eskd"] = (
    ((diagnoses["icd_version"] == 9) & (diagnoses["icd_code"] == "5856"))
    | ((diagnoses["icd_version"] == 10) & (diagnoses["icd_code"] == "N186"))
)

# Flag chronic dialysis status codes
diagnoses["dialysis"] = (
    ((diagnoses["icd_version"] == 9)
     & diagnoses["icd_code"].isin(["V451", "V4511"]))
    | ((diagnoses["icd_version"] == 10)
       & (diagnoses["icd_code"] == "Z992"))
)

# Keep records with either flag
renal_diagnoses = diagnoses[
    diagnoses["eskd"] | diagnoses["dialysis"]
].copy()

# Summarise the matching codes
print("Matching diagnosis records:", len(renal_diagnoses))
print("Patients with matching codes:", renal_diagnoses["subject_id"].nunique())

print("\nRecords per code:")
print(
    renal_diagnoses.groupby(["icd_version", "icd_code"]).size()
)

Matching diagnosis records: 19108
Patients with matching codes: 2686

Records per code:
icd_version  icd_code
9            5856        6260
             V451          61
             V4511       3783
10           N186        4893
             Z992        4111
dtype: int64


Checking if renal diagnoses was before ED arrival

In [22]:
# Add discharge times to the matching diagnosis records
renal_history = renal_diagnoses.merge(
    admissions_raw[["subject_id", "hadm_id", "dischtime"]],
    on=["subject_id", "hadm_id"],
    how="left",
    validate="many_to_one"
)

# Rename columns to distinguish historical and current admissions
renal_history = renal_history.rename(columns={
    "hadm_id": "history_hadm_id",
    "dischtime": "history_dischtime"
})

renal_history["history_dischtime"] = pd.to_datetime(
    renal_history["history_dischtime"]
)

print("Missing historical discharge times:",
      renal_history["history_dischtime"].isna().sum())

# Match each ED encounter to the patient's renal diagnosis history
history_check = cohort[
    ["subject_id", "stay_id", "hadm_id", "intime"]
].merge(
    renal_history,
    on="subject_id",
    how="inner"
)

# Keep evidence from a different admission completed before ED arrival
prior_history = history_check[
    (history_check["history_hadm_id"] != history_check["hadm_id"])
    & (history_check["history_dischtime"] < history_check["intime"])
].copy()

# Get the ED stays with each type of prior evidence
eskd_stays = prior_history.loc[
    prior_history["eskd"], "stay_id"
].unique()

dialysis_stays = prior_history.loc[
    prior_history["dialysis"], "stay_id"
].unique()

# Add encounter-level flags
cohort["prior_eskd"] = cohort["stay_id"].isin(eskd_stays)
cohort["prior_dialysis"] = cohort["stay_id"].isin(dialysis_stays)

# Count encounters, not diagnosis records
print("Prior ESKD:", cohort["prior_eskd"].sum())
print("Prior dialysis:", cohort["prior_dialysis"].sum())

print("Both:",
      (cohort["prior_eskd"] & cohort["prior_dialysis"]).sum())

print("Either:",
      (cohort["prior_eskd"] | cohort["prior_dialysis"]).sum())

Missing historical discharge times: 0
Prior ESKD: 7504
Prior dialysis: 5951
Both: 5821
Either: 7634


Applying renal exclusions

In [23]:
# Flag encounters to exclude
renal_exclusion = cohort["prior_eskd"] | cohort["prior_dialysis"]

# Keep excluded encounters separately for our records
renal_excluded = cohort[renal_exclusion].copy()

# Keep encounters without either exclusion
cohort = cohort[~renal_exclusion].copy()

# Check the remaining cohort
print("Excluded encounters:", len(renal_excluded))
print("Remaining encounters:", len(cohort))
print("Unique ED stays:", cohort["stay_id"].nunique())
print("Unique patients:", cohort["subject_id"].nunique())

Excluded encounters: 7634
Remaining encounters: 195340
Unique ED stays: 195340
Unique patients: 106516


Define outcome observation window. Starts at ED arrival and ends at 24hr or hospital discharge.

In [24]:
# Start with 24 hours after ED arrival
cohort["endpoint_time"] = cohort["intime"] + pd.Timedelta(hours=24)

# Shorten the window if hospital discharge occurs earlier
early_discharge = cohort["dischtime"] < cohort["endpoint_time"]

cohort.loc[early_discharge, "endpoint_time"] = cohort.loc[
    early_discharge, "dischtime"
]

# Calculate the available follow-up in hours
cohort["followup_hours"] = (
    cohort["endpoint_time"] - cohort["intime"]
).dt.total_seconds() / 3600

# Inspect the observation windows
print("Discharged before 24 hours:", early_discharge.sum())
print("Full 24-hour windows:", (cohort["followup_hours"] == 24).sum())

display(cohort["followup_hours"].describe())

Discharged before 24 hours: 41839
Full 24-hour windows: 153501


,followup_hours
count,195340.000000
mean,22.128806
std,4.414654
min,0.200000
25%,24.000000
50%,24.000000
75%,24.000000
max,24.000000


# Creatinine extraction and exploration

In [25]:
# Get the laboratory item dictionary
query = f"""
SELECT *
FROM `{HOSP}.d_labitems`
"""

lab_items = client.query(query).result().to_dataframe(
    create_bqstorage_client=False
)

# Find items with creatinine in their name
creatinine_items = lab_items[
    lab_items["label"].str.contains("creatinine", case=False, na=False)
]

# Inspect item names and specimen types
display(
    creatinine_items[
        ["itemid", "label", "fluid", "category"]
    ].sort_values("itemid")
)

,itemid,label,fluid,category
636,50841,"Creatinine, Ascites",Ascites,Chemistry
124,50912,Creatinine,Blood,Chemistry
692,51021,"Creatinine, Joint Fluid",Joint Fluid,Chemistry
724,51032,"Creatinine, Body Fluid",Other Body Fluid,Chemistry
665,51052,"Creatinine, Pleural",Pleural,Chemistry
520,51067,24 hr Creatinine,Urine,Chemistry
523,51070,"Albumin/Creatinine, Urine",Urine,Chemistry
526,51073,"Amylase/Creatinine Ratio, Urine",Urine,Chemistry
533,51080,Creatinine Clearance,Urine,Chemistry
534,51081,"Creatinine, Serum",Urine,Chemistry


Get creatinine measurements

In [26]:
# Get patients remaining in the eligible cohort
cohort_patient_ids = cohort["subject_id"].unique().tolist()

# Download raw creatinine records for these patients
query = f"""
SELECT labevent_id, subject_id, hadm_id, specimen_id,
       itemid, charttime, storetime, value, valuenum, valueuom
FROM `{HOSP}.labevents`
WHERE subject_id IN UNNEST(@patient_ids)
  AND itemid IN (50912, 52546, 52024)
"""

job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ArrayQueryParameter(
            "patient_ids", "INT64", cohort_patient_ids
        )
    ]
)

creatinine_raw = client.query(
    query, job_config=job_config
).result().to_dataframe(create_bqstorage_client=False)

# Check the downloaded records
print("Creatinine records:", len(creatinine_raw))
print("Unique lab events:", creatinine_raw["labevent_id"].nunique())
print("Unique patients:", creatinine_raw["subject_id"].nunique())

print("\nRecords by item:")
print(creatinine_raw["itemid"].value_counts())

print("\nRecorded units:")
print(creatinine_raw["valueuom"].value_counts(dropna=False))

print("\nMissing values:")
print(creatinine_raw.isna().sum())

Creatinine records: 2211428
Unique lab events: 2211428
Unique patients: 104103

Records by item:
itemid
50912    2207518
52024       3648
52546        262
Name: count, dtype: Int64

Recorded units:
valueuom
mg/dL    2211428
Name: count, dtype: int64

Missing values:
labevent_id         0
subject_id          0
hadm_id        888678
specimen_id         0
itemid              0
charttime           0
storetime           2
value             735
valuenum          735
valueuom            0
dtype: int64


In [27]:
# Keep the two chemistry items for the primary definition
creatinine = creatinine_raw[
    creatinine_raw["itemid"].isin([50912, 52546])
].copy()

# Keep whole-blood measurements separately
creatinine_whole_blood = creatinine_raw[
    creatinine_raw["itemid"] == 52024
].copy()

# Convert sample times
creatinine["charttime"] = pd.to_datetime(creatinine["charttime"])

# Use a numeric copy for exploration only
creatinine_values = pd.to_numeric(
    creatinine["valuenum"], errors="coerce"
)

print("Primary creatinine records:", len(creatinine))
print("Missing numeric results:", creatinine_values.isna().sum())

# Inspect the distribution
display(creatinine_values.describe())

print("\nLowest 10 distinct values:")
print(creatinine_values.dropna().sort_values().unique()[:10])

print("\nHighest 10 distinct values:")
print(creatinine_values.dropna().sort_values().unique()[-10:])

# Count records sharing a patient and sample time
same_time = creatinine.duplicated(
    subset=["subject_id", "charttime"], keep=False
)

print("\nRecords sharing a patient and sample time:", same_time.sum())

Primary creatinine records: 2207780
Missing numeric results: 728


,valuenum
count,2.207052e+06
mean,1.265740e+00
std,1.219128e+00
min,0.000000e+00
25%,7.000000e-01
50%,9.000000e-01
75%,1.300000e+00
max,8.000000e+01



Lowest 10 distinct values:
[0.   0.07 0.08 0.1  0.2  0.3  0.4  0.5  0.6  0.7 ]

Highest 10 distinct values:
[36.9 38.  38.1 38.3 41.5 41.7 43.  62.5 74.  80. ]

Records sharing a patient and sample time: 4884


Screen values anad check same time conflicts. To retain extraction rule of greater than zero and no more than 150 mg/dL

In [28]:
# Flag values passing the extraction screen
valid_value = (creatinine_values > 0) & (creatinine_values <= 150)

print("Missing numeric results:", creatinine_values.isna().sum())
print("Zero or negative results:", (creatinine_values <= 0).sum())
print("Results above 150:", (creatinine_values > 150).sum())

# Keep usable records without changing the original values
creatinine_valid = creatinine[valid_value].copy()

# Count different values at each patient and sample time
values_per_time = creatinine_valid.groupby(
    ["subject_id", "charttime"]
)["valuenum"].nunique()

print("\nUsable records:", len(creatinine_valid))
print("Unique patient/sample times:", len(values_per_time))
print("Times with conflicting values:", (values_per_time > 1).sum())

Missing numeric results: 728
Zero or negative results: 102
Results above 150: 0

Usable records: 2206950
Unique patient/sample times: 2204521
Times with conflicting values: 960


Removing conflicting samples

In [29]:
# Count distinct values at each patient and sample time
value_counts = creatinine_valid.groupby(
    ["subject_id", "charttime"]
)["valuenum"].transform("nunique")

# Keep conflicting records separately for review
creatinine_conflicts = creatinine_valid[value_counts > 1].copy()

# Keep records where the values agree
creatinine_clean = creatinine_valid[value_counts == 1].copy()

print("Records excluded for conflicts:", len(creatinine_conflicts))
print("Remaining lab records:", len(creatinine_clean))

# Count the remaining distinct sample times
sample_times = creatinine_clean[
    ["subject_id", "charttime"]
].drop_duplicates()

print("Remaining patient/sample times:", len(sample_times))

Records excluded for conflicts: 1926
Remaining lab records: 2205024
Remaining patient/sample times: 2203561


Preparing decimal creatinine values

In [30]:
from decimal import Decimal

# Preserve exact decimal values for outcome calculations
creatinine_clean["creat_decimal"] = (
    creatinine_clean["valuenum"].astype(str).map(Decimal)
)

# Check the number of prepared values
print("Decimal values prepared:", creatinine_clean["creat_decimal"].notna().sum())

# Demonstrate an exact threshold calculation
difference = Decimal("0.9") - Decimal("0.6")

print("0.9 minus 0.6:", difference)
print("Meets the 0.3 threshold:", difference >= Decimal("0.3"))

Decimal values prepared: 2205024
0.9 minus 0.6: 0.3
Meets the 0.3 threshold: True


Linking creatinine records to ED observation windows

In [31]:
# Select the encounter fields needed for linkage
windows = cohort[
    ["subject_id", "stay_id", "hadm_id", "intime", "endpoint_time"]
].copy()

windows = windows.rename(columns={"hadm_id": "ed_hadm_id"})

# Select the laboratory fields needed
labs = creatinine_clean[
    ["labevent_id", "subject_id", "hadm_id", "charttime", "creat_decimal"]
].copy()

# Process 5,000 patients at a time
patient_ids = cohort["subject_id"].unique()
matched_parts = []

for start in range(0, len(patient_ids), 5000):
    batch_ids = patient_ids[start:start + 5000]

    batch_labs = labs[labs["subject_id"].isin(batch_ids)]
    batch_windows = windows[windows["subject_id"].isin(batch_ids)]

    matches = batch_labs.merge(batch_windows, on="subject_id")

    # Require the sample to fall inside the observation window
    within_window = (
        (matches["charttime"] >= matches["intime"])
        & (matches["charttime"] <= matches["endpoint_time"])
    )

    # Accept matching or missing laboratory admission IDs
    admission_match = (
        matches["hadm_id"].isna()
        | (matches["hadm_id"] == matches["ed_hadm_id"])
    )

    matched_parts.append(
        matches[within_window & admission_match].copy()
    )

# Combine the matched batches
current_matches = pd.concat(matched_parts, ignore_index=True)

print("Lab–encounter matches:", len(current_matches))
print("Unique lab events:", current_matches["labevent_id"].nunique())
print("Encounters with measurements:", current_matches["stay_id"].nunique())

# Check whether a lab event matches multiple ED encounters
encounters_per_event = current_matches.groupby("labevent_id")["stay_id"].nunique()

print("Lab events matching multiple encounters:",
      (encounters_per_event > 1).sum())

Lab–encounter matches: 334473
Unique lab events: 334015
Encounters with measurements: 185443
Lab events matching multiple encounters: 456


Dealing with overlapping matches and repeated sample times

In [32]:
# Put the most recent ED arrival first for each lab event
sorted_matches = current_matches.sort_values(
    ["labevent_id", "intime", "stay_id"],
    ascending=[True, False, True]
)

# Assign each lab event to one encounter
current_events = sorted_matches.drop_duplicates(
    subset="labevent_id", keep="first"
).copy()

print("Matches removed:", len(current_matches) - len(current_events))
print("Assigned lab events:", len(current_events))

# Count agreeing records at the same time only once per encounter
current_samples = current_events.drop_duplicates(
    subset=["stay_id", "charttime"]
).copy()

# Count distinct sample times for each encounter
sample_counts = current_samples.groupby("stay_id").size()

cohort["current_sample_count"] = (
    cohort["stay_id"].map(sample_counts).fillna(0).astype(int)
)

print("Distinct encounter/sample times:", len(current_samples))

print("\nEncounters without samples:",
      (cohort["current_sample_count"] == 0).sum())

print("Encounters with one sample:",
      (cohort["current_sample_count"] == 1).sum())

print("Encounters with multiple samples:",
      (cohort["current_sample_count"] >= 2).sum())

Matches removed: 458
Assigned lab events: 334015
Distinct encounter/sample times: 333909

Encounters without samples: 10072
Encounters with one sample: 72555
Encounters with multiple samples: 112713


Summarise sampling times

In [33]:
# Find the first and last sample time for each encounter
sampling = current_samples.groupby("stay_id")["charttime"].agg(
    ["min", "max"]
)

sampling = sampling.rename(columns={
    "min": "first_sample_time",
    "max": "last_sample_time"
})

# Add these times to the cohort
cohort = cohort.join(sampling, on="stay_id")

# Time from ED arrival to the first sample
cohort["first_sample_delay_hours"] = (
    cohort["first_sample_time"] - cohort["intime"]
).dt.total_seconds() / 3600

# Time between the first and last samples
cohort["sampling_span_hours"] = (
    cohort["last_sample_time"] - cohort["first_sample_time"]
).dt.total_seconds() / 3600

# Time from the last sample to the end of follow-up
cohort["endpoint_gap_hours"] = (
    cohort["endpoint_time"] - cohort["last_sample_time"]
).dt.total_seconds() / 3600

# Summarise encounters with at least two sample times
serial = cohort[cohort["current_sample_count"] >= 2]

display(serial[
    ["first_sample_delay_hours", "sampling_span_hours", "endpoint_gap_hours"]
].describe())

,first_sample_delay_hours,sampling_span_hours,endpoint_gap_hours
count,112713.000000,112713.000000,112713.000000
mean,1.277556,14.609989,7.845118
std,1.377181,4.773699,4.510218
min,0.000000,0.016667,0.000000
25%,0.433333,11.166667,4.366667
50%,0.850000,14.916667,7.383333
75%,1.666667,18.366667,10.883333
max,21.216667,23.950000,23.916667


# Outcome Comparisions

Preparing reference measurements

In [34]:
# Keep one agreeing value per patient and sample time
reference_samples = creatinine_clean[
    ["subject_id", "charttime", "creat_decimal"]
].drop_duplicates(
    subset=["subject_id", "charttime"]
).copy()

# Sort each patient's measurements by time
reference_samples = reference_samples.sort_values(
    ["subject_id", "charttime"]
)

# Rename fields to distinguish reference and current measurements
reference_samples = reference_samples.rename(columns={
    "charttime": "reference_time",
    "creat_decimal": "reference_creatinine"
})

print("Reference sample times:", len(reference_samples))
print("Patients with reference measurements:",
      reference_samples["subject_id"].nunique())

print("Duplicate patient/sample times:",
      reference_samples.duplicated(
          subset=["subject_id", "reference_time"]
      ).sum())

Reference sample times: 2203561
Patients with reference measurements: 104052
Duplicate patient/sample times: 0


Finding the minimun earlier creatinine within 48hrs or 7 days for each current sample

In [35]:
# Select the current samples to assess
samples = current_samples[
    ["subject_id", "stay_id", "charttime", "creat_decimal"]
].copy()

# Give each sample a unique row number
samples = samples.reset_index(drop=True)
samples["sample_id"] = samples.index

patient_ids = samples["subject_id"].unique()
comparison_parts = []

for start in range(0, len(patient_ids), 1000):
    batch_ids = patient_ids[start:start + 1000]

    batch_samples = samples[samples["subject_id"].isin(batch_ids)]
    batch_history = reference_samples[
        reference_samples["subject_id"].isin(batch_ids)
    ]

    pairs = batch_samples.merge(batch_history, on="subject_id")

    # Calculate how long before the current sample each reference occurred
    time_gap = pairs["charttime"] - pairs["reference_time"]

    # Keep strictly earlier references within seven days
    earlier_7d = (
        (time_gap > pd.Timedelta(0))
        & (time_gap <= pd.Timedelta(days=7))
    )

    pairs = pairs[earlier_7d].copy()

    # Find the minimum reference within seven days
    minimum_7d = pairs.groupby("sample_id")["reference_creatinine"].min()

    # Find the minimum reference within 48 hours
    within_48h = (
        pairs["charttime"] - pairs["reference_time"]
    ) <= pd.Timedelta(hours=48)

    minimum_48h = pairs[within_48h].groupby(
        "sample_id"
    )["reference_creatinine"].min()

    # Add the references to the current samples
    batch_samples = batch_samples.copy()
    batch_samples["minimum_48h"] = batch_samples["sample_id"].map(minimum_48h)
    batch_samples["minimum_7d"] = batch_samples["sample_id"].map(minimum_7d)

    comparison_parts.append(batch_samples)

# Combine the batches
sample_comparisons = pd.concat(comparison_parts, ignore_index=True)

print("Current samples:", len(sample_comparisons))
print("With a 48-hour reference:",
      sample_comparisons["minimum_48h"].notna().sum())
print("With a seven-day reference:",
      sample_comparisons["minimum_7d"].notna().sum())
print("Without either reference:",
      sample_comparisons["minimum_7d"].isna().sum())

Current samples: 333909
With a 48-hour reference: 160645
With a seven-day reference: 177140
Without either reference: 156769


Flag qualifying rises

In [36]:
# Create a working copy
sample_results = sample_comparisons.copy()

# Start with no observed qualifying rise
sample_results["rise_48h"] = False
sample_results["rise_7d"] = False

# Check the absolute rise where a 48-hour reference exists
has_48h = sample_results["minimum_48h"].notna()

sample_results.loc[has_48h, "rise_48h"] = (
    sample_results.loc[has_48h, "creat_decimal"]
    - sample_results.loc[has_48h, "minimum_48h"]
) >= Decimal("0.3")

# Check the relative rise where a seven-day reference exists
has_7d = sample_results["minimum_7d"].notna()

sample_results.loc[has_7d, "rise_7d"] = (
    sample_results.loc[has_7d, "creat_decimal"]
    >= sample_results.loc[has_7d, "minimum_7d"] * Decimal("1.5")
)

# Either criterion identifies a positive sample
sample_results["positive_sample"] = (
    sample_results["rise_48h"] | sample_results["rise_7d"]
)

# Keep reference availability explicit
sample_results["has_comparator"] = has_7d

# Count positive samples and encounters
positive_samples = sample_results[sample_results["positive_sample"]]

print("Samples meeting the 48-hour criterion:",
      sample_results["rise_48h"].sum())

print("Samples meeting the seven-day criterion:",
      sample_results["rise_7d"].sum())

print("Samples meeting either criterion:", len(positive_samples))
print("Encounters with an observed rise:",
      positive_samples["stay_id"].nunique())

Samples meeting the 48-hour criterion: 8589
Samples meeting the seven-day criterion: 5890
Samples meeting either criterion: 11920
Encounters with an observed rise: 7866


Add observed positive flags and first qualifying times

In [37]:
# Identify encounters with at least one positive sample
positive_stays = positive_samples["stay_id"].unique()

cohort["observed_positive"] = cohort["stay_id"].isin(positive_stays)

# Find the first qualifying sample time for each positive encounter
first_positive_time = positive_samples.groupby("stay_id")["charttime"].min()

cohort["first_positive_time"] = cohort["stay_id"].map(first_positive_time)

# Calculate time from ED arrival to the first qualifying sample
cohort["hours_to_first_positive"] = (
    cohort["first_positive_time"] - cohort["intime"]
).dt.total_seconds() / 3600

# Check that positive times fall inside the observation window
bad_positive_times = cohort["observed_positive"] & (
    (cohort["first_positive_time"] < cohort["intime"])
    | (cohort["first_positive_time"] > cohort["endpoint_time"])
)

print("Observed positive encounters:", cohort["observed_positive"].sum())
print("Positive times outside the window:", bad_positive_times.sum())

# Summarise when qualifying rises were first identified
display(cohort["hours_to_first_positive"].describe())

Observed positive encounters: 7866
Positive times outside the window: 0


,hours_to_first_positive
count,7866.000000
mean,10.218752
std,7.770730
min,0.000000
25%,1.500000
50%,10.675000
75%,17.083333
max,24.000000


Check the first current sample

In [38]:
# Select the first current sample for each encounter
first_samples = sample_results.sort_values("charttime").drop_duplicates(
    subset="stay_id"
).copy()

# Start with an unassessable first sample
first_samples["first_sample_status"] = "unassessable"

# A comparator exists, but no qualifying rise was found
assessable_no_rise = (
    first_samples["has_comparator"]
    & ~first_samples["positive_sample"]
)

first_samples.loc[
    assessable_no_rise, "first_sample_status"
] = "no qualifying rise"

# The first sample already meets a positive criterion
first_samples.loc[
    first_samples["positive_sample"], "first_sample_status"
] = "positive"

# Add the status to the cohort
first_status = first_samples.set_index("stay_id")["first_sample_status"]

cohort["first_sample_status"] = cohort["stay_id"].map(first_status)
cohort["first_sample_status"] = cohort["first_sample_status"].fillna(
    "no current sample"
)

# Inspect first-sample status among positive encounters
print("First-sample status among positive encounters:")
print(
    cohort.loc[
        cohort["observed_positive"], "first_sample_status"
    ].value_counts()
)

First-sample status among positive encounters:
first_sample_status
unassessable          4514
positive              2627
no qualifying rise     725
Name: count, dtype: int64


Calculate observed creatinine stages

In [39]:
# Leave samples without an observed rise unstaged
sample_results["creatinine_stage"] = pd.Series(
    pd.NA, index=sample_results.index, dtype="Int64"
)

# Start positive samples at stage 1
sample_results.loc[
    sample_results["positive_sample"], "creatinine_stage"
] = 1

# Check the higher ratio thresholds where a reference exists
has_reference = sample_results["minimum_7d"].notna()

current = sample_results.loc[has_reference, "creat_decimal"]
reference = sample_results.loc[has_reference, "minimum_7d"]

stage2_index = current[current >= reference * Decimal("2")].index
stage3_index = current[current >= reference * Decimal("3")].index

sample_results.loc[stage2_index, "creatinine_stage"] = 2
sample_results.loc[stage3_index, "creatinine_stage"] = 3

# The absolute stage 3 route also requires a qualifying acute rise
absolute_stage3 = (
    sample_results["positive_sample"]
    & (sample_results["creat_decimal"] >= Decimal("4"))
)

sample_results.loc[absolute_stage3, "creatinine_stage"] = 3

# Find the highest observed stage in each encounter
highest_stage = sample_results.groupby("stay_id")["creatinine_stage"].max()

cohort["highest_creatinine_stage"] = cohort["stay_id"].map(highest_stage)

print("Highest stage among positive encounters:")
print(
    cohort.loc[
        cohort["observed_positive"], "highest_creatinine_stage"
    ].value_counts().sort_index()
)

print("\nStage 2 or higher:",
      (cohort["highest_creatinine_stage"] >= 2).sum())

Highest stage among positive encounters:
highest_creatinine_stage
1    5612
2     720
3    1534
Name: count, dtype: Int64

Stage 2 or higher: 2254


Record the first stage >2 sample time for the secondary outcome

In [40]:
# Select samples meeting stage 2 or 3
severe_samples = sample_results[
    sample_results["creatinine_stage"] >= 2
].copy()

# Find the first qualifying time for each encounter
first_stage2_time = severe_samples.groupby("stay_id")["charttime"].min()

cohort["first_stage2_time"] = cohort["stay_id"].map(first_stage2_time)
cohort["observed_stage2_positive"] = cohort["first_stage2_time"].notna()

# Check that secondary positives are also primary positives
invalid_secondary = (
    cohort["observed_stage2_positive"]
    & ~cohort["observed_positive"]
)

# Stage 2 cannot be identified before the first qualifying rise
invalid_order = (
    cohort["first_stage2_time"] < cohort["first_positive_time"]
)

print("Stage 2 or higher encounters:",
      cohort["observed_stage2_positive"].sum())

print("Secondary positives without a primary positive:",
      invalid_secondary.sum())

print("Stage 2 time before first positive time:",
      invalid_order.sum())

Stage 2 or higher encounters: 2254
Secondary positives without a primary positive: 0
Stage 2 time before first positive time: 0


# Control assessment

Identify encounters with serial measurements and no observed rise

In [41]:
# Check whether each encounter has any usable timed comparison
has_comparison = sample_results.groupby("stay_id")["has_comparator"].any()

cohort["has_timed_comparison"] = (
    cohort["stay_id"].map(has_comparison).eq(True)
)

# Identify potential controls before checking older historical values
cohort["serial_no_rise"] = (
    (cohort["current_sample_count"] >= 2)
    & cohort["has_timed_comparison"]
    & ~cohort["observed_positive"]
)

print("Serial measurements with no observed rise:",
      cohort["serial_no_rise"].sum())

# Inspect whether the first sample was assessable
print("\nFirst-sample status in these potential controls:")
print(
    cohort.loc[
        cohort["serial_no_rise"], "first_sample_status"
    ].value_counts()
)

Serial measurements with no observed rise: 105231

First-sample status in these potential controls:
first_sample_status
unassessable          89198
no qualifying rise    16033
Name: count, dtype: int64


Assign outcome states and binary labels \
Operational negative definition: At least two usable current sample times, a usable timed comparison, and no observed qualifying creatinine rise.

In [42]:
# Start with indeterminate
cohort["label_state"] = "indeterminate"

# No usable creatinine within the observation window
cohort.loc[
    cohort["current_sample_count"] == 0, "label_state"
] = "no_usable_current_creatinine"

# Serial measurements with a timed comparison and no qualifying rise
cohort.loc[
    cohort["serial_no_rise"], "label_state"
] = "negative"

# Any observed qualifying rise
cohort.loc[
    cohort["observed_positive"], "label_state"
] = "positive"

# Assign binary labels only to positives and operational negatives
cohort["aki_label"] = pd.Series(
    pd.NA, index=cohort.index, dtype="Int64"
)

cohort.loc[cohort["label_state"] == "negative", "aki_label"] = 0
cohort.loc[cohort["label_state"] == "positive", "aki_label"] = 1

# Check the counts
print("Outcome states:")
print(cohort["label_state"].value_counts())

print("\nLabelled encounters:", cohort["aki_label"].notna().sum())

Outcome states:
label_state
negative                        105231
indeterminate                    72171
no_usable_current_creatinine     10072
positive                          7866
Name: count, dtype: int64

Labelled encounters: 113097


Check outcome label consistency

In [43]:
# Identify each labelled group
positive = cohort["label_state"] == "positive"
negative = cohort["label_state"] == "negative"
unresolved = cohort["label_state"].isin(
    ["indeterminate", "no_usable_current_creatinine"]
)

# Check that negatives satisfy the agreed rule
invalid_negatives = negative & (
    (cohort["current_sample_count"] < 2)
    | ~cohort["has_timed_comparison"]
    | cohort["observed_positive"]
)

print("Invalid negatives:", invalid_negatives.sum())

# Check that positive labels match observed rises
print("Positive flag disagreements:",
      (positive != cohort["observed_positive"]).sum())

# Check that unresolved encounters have no binary label
print("Unresolved encounters with a binary label:",
      cohort.loc[unresolved, "aki_label"].notna().sum())

print("Labelled encounters missing a binary label:",
      cohort.loc[positive | negative, "aki_label"].isna().sum())

# Check encounter uniqueness
print("Duplicate ED stays:", cohort["stay_id"].duplicated().sum())

# Summarise the labelled population
labelled = cohort[cohort["aki_label"].notna()].copy()

print("\nLabelled encounters:", len(labelled))
print("Labelled patients:", labelled["subject_id"].nunique())
print("Positive percentage:",
      round(labelled["aki_label"].mean() * 100, 2))

Invalid negatives: 0
Positive flag disagreements: 0
Unresolved encounters with a binary label: 0
Labelled encounters missing a binary label: 0
Duplicate ED stays: 0

Labelled encounters: 113097
Labelled patients: 66090
Positive percentage: 6.96


# Predictor preparation

Inspect predictors in the labelled cohort

In [44]:
# Create a working copy of the labelled cohort
model_data = labelled.copy()

# Select the predictor fields
predictors = [
    "age", "gender", "arrival_transport",
    "temperature", "heartrate", "resprate",
    "o2sat", "sbp", "dbp",
    "acuity", "pain", "chiefcomplaint"
]

# Check missing predictor values
print("Missing predictor values:")
print(model_data[predictors].isna().sum())

# Inspect vital signs before cleaning
vitals = ["temperature", "heartrate", "resprate", "o2sat", "sbp", "dbp"]

display(model_data[vitals].describe())

# Count encounters missing all six vital signs
print("All six vital signs missing:",
      model_data[vitals].isna().all(axis=1).sum())

Missing predictor values:
age                      0
gender                   0
arrival_transport        0
temperature          11876
heartrate             9428
resprate             10634
o2sat                10698
sbp                   9682
dbp                   9951
acuity                3759
pain                  6477
chiefcomplaint           7
dtype: int64


,temperature,heartrate,resprate,o2sat,sbp,dbp
count,101221.000000,103669.000000,102463.000000,102399.000000,103415.000000,103146.000000
mean,98.099967,87.271169,18.176708,97.874161,133.597183,78.154173
std,2.868239,20.287787,3.239234,30.144470,38.895529,327.834964
min,0.100000,1.000000,1.000000,0.000000,1.000000,0.000000
25%,97.600000,73.000000,16.000000,97.000000,116.000000,64.000000
50%,98.100000,86.000000,18.000000,98.000000,132.000000,74.000000
75%,98.600000,100.000000,19.000000,100.000000,149.000000,85.000000
max,148.740000,978.000000,187.000000,9322.000000,9656.000000,74810.000000


All six vital signs missing: 8823


Inspect unusually high values

In [45]:
# Inspect the frequency of extreme values
for column in vitals:
    counts = model_data[column].value_counts().sort_index()

    print("\n", column)

    print("Lowest values and their counts:")
    print(counts.head(10))

    print("Highest values and their counts:")
    print(counts.tail(10))


 temperature
Lowest values and their counts:
temperature
0.1    1
0.3    1
0.5    1
6.0    1
7.2    1
7.6    1
7.8    3
7.9    2
8.4    1
8.6    1
Name: count, dtype: int64
Highest values and their counts:
temperature
105.60    3
105.70    1
105.80    2
105.90    1
106.00    1
106.30    1
106.60    2
107.00    1
111.40    1
148.74    1
Name: count, dtype: int64

 heartrate
Lowest values and their counts:
heartrate
1.0     1
8.0     1
9.0     1
14.0    1
16.0    2
18.0    4
20.0    3
24.0    2
25.0    5
26.0    1
Name: count, dtype: int64
Highest values and their counts:
heartrate
185.0    1
187.0    2
189.0    1
190.0    3
200.0    6
221.0    1
227.0    1
230.0    1
570.0    1
978.0    1
Name: count, dtype: int64

 resprate
Lowest values and their counts:
resprate
1.0       3
2.0       5
4.0       2
6.0       1
8.0      27
9.0       2
10.0     89
11.0     16
12.0    690
13.0    101
Name: count, dtype: int64
Highest values and their counts:
resprate
65.0     1
72.0     1
80.0     2
96.

Clean vital signs and acuity. \
These are broad study screening limits, not normal clinical ranges. \
Temperature values from 30 - 43 are assumed to be Celsius and converted to Fahrenheit

In [46]:
# Preserve the original values so this block can be rerun
columns = vitals + ["acuity"]

for column in columns:
    model_data[column + "_raw"] = labelled[column]

# Clean temperature
temp = model_data["temperature_raw"]

model_data["temperature"] = temp.where((temp > 70) & (temp < 120))

# Convert assumed Celsius values to Fahrenheit
celsius = temp.between(30, 43)

model_data.loc[celsius, "temperature"] = (
    temp[celsius] * 9 / 5 + 32
)

# Clean the other vital signs using the original limits
limits = {
    "heartrate": 300,
    "resprate": 70,
    "sbp": 400,
    "dbp": 300
}

for column, upper_limit in limits.items():
    values = model_data[column + "_raw"]
    model_data[column] = values.where(
        (values > 0) & (values < upper_limit)
    )

# Oxygen saturation allows 100
oxygen = model_data["o2sat_raw"]
model_data["o2sat"] = oxygen.where((oxygen > 0) & (oxygen <= 100))

# Keep acuity values from 1 to 5
acuity = pd.to_numeric(model_data["acuity_raw"], errors="coerce")
model_data["acuity"] = acuity.where(acuity.between(1, 5))

# Check the changes
print("Temperature values converted:", celsius.sum())

print("\nAdditional missing values after cleaning:")
print(
    model_data[columns].isna().sum()
    - labelled[columns].isna().sum()
)

display(model_data[vitals].describe())

Temperature values converted: 129

Additional missing values after cleaning:
temperature     33
heartrate        2
resprate        11
o2sat           14
sbp              3
dbp            116
acuity           0
dtype: int64


,temperature,heartrate,resprate,o2sat,sbp,dbp
count,101188.000000,103667.000000,102452.000000,102385.000000,103412.000000,103030.000000
mean,98.202108,87.257920,18.166439,97.709325,133.492919,74.645132
std,1.154698,20.042475,3.059818,3.040007,25.054443,16.314075
min,72.000000,1.000000,1.000000,2.000000,1.000000,1.000000
25%,97.600000,73.000000,16.000000,97.000000,116.000000,64.000000
50%,98.100000,86.000000,18.000000,98.000000,132.000000,74.000000
75%,98.612500,100.000000,19.000000,100.000000,149.000000,85.000000
max,111.400000,230.000000,65.000000,100.000000,274.000000,280.000000


Preparing pain and chief complaint

In [47]:
# Preserve the original text
model_data["pain_raw"] = labelled["pain"]
model_data["chiefcomplaint_raw"] = labelled["chiefcomplaint"]

# Remove spaces around pain entries
pain = model_data["pain_raw"].astype("string").str.strip()

# Accept 0–10, including entries such as "5.0"
numeric_pain = pain.str.fullmatch(r"([0-9]|10)(\.0+)?", na=False)

model_data["pain_numeric"] = pd.to_numeric(
    pain.where(numeric_pain), errors="coerce"
)

# Distinguish missing pain from other recorded entries
missing_pain = pain.isna() | pain.eq("").fillna(False)

model_data["pain_status"] = "other_recorded"
model_data.loc[missing_pain, "pain_status"] = "missing"
model_data.loc[numeric_pain, "pain_status"] = "numeric"

# Trim chief complaints and treat blank text as missing
complaint = model_data["chiefcomplaint_raw"].astype("string").str.strip()
model_data["chiefcomplaint"] = complaint.replace("", pd.NA)

# Check the results
print("Pain status:")
print(model_data["pain_status"].value_counts())

print("\nNumeric pain summary:")
print(model_data["pain_numeric"].describe())

print("\nMissing chief complaints:",
      model_data["chiefcomplaint"].isna().sum())

print("\nMost common other pain entries:")
print(pain[model_data["pain_status"] == "other_recorded"].value_counts().head(15))

Pain status:
pain_status
numeric           95008
other_recorded    11612
missing            6477
Name: count, dtype: int64

Numeric pain summary:
count     95008.0
mean     3.421186
std      3.735968
min           0.0
25%           0.0
50%           2.0
75%           7.0
max          10.0
Name: pain_numeric, dtype: Float64

Missing chief complaints: 7

Most common other pain entries:
pain_raw
13          4236
unable      1811
Critical     995
UTA          863
uta          834
ua           583
UA           238
c            231
u/a          197
Unable       139
crit          87
ett           71
critical      69
?             48
u             42
Name: count, dtype: Int64


Summarize cleaned predictor missingness

In [48]:
# List the cleaned predictors
predictor_columns = [
    "age", "gender", "arrival_transport",
    "temperature", "heartrate", "resprate",
    "o2sat", "sbp", "dbp",
    "acuity", "pain_numeric", "pain_status", "chiefcomplaint"
]

# Show missing counts and percentages
missing_summary = pd.DataFrame({
    "missing_count": model_data[predictor_columns].isna().sum(),
    "missing_percent": (
        model_data[predictor_columns].isna().mean() * 100
    ).round(2)
})

display(missing_summary)

# Check how many encounters lack all six vital signs
all_vitals_missing = model_data[vitals].isna().all(axis=1)

print("All six vital signs missing:", all_vitals_missing.sum())

# Count complete cases for comparison only
complete_cases = model_data[predictor_columns].notna().all(axis=1)

print("Complete predictor records:", complete_cases.sum())
print("Total encounters retained:", len(model_data))

,missing_count,missing_percent
age,0,0.00
gender,0,0.00
arrival_transport,0,0.00
temperature,11909,10.53
heartrate,9430,8.34
resprate,10645,9.41
o2sat,10712,9.47
sbp,9685,8.56
dbp,10067,8.90
acuity,3759,3.32


All six vital signs missing: 8823
Complete predictor records: 89902
Total encounters retained: 113097


Seperate model inputs from outcomes and identifiers

In [49]:
# Keep only the intended predictors
X = model_data[predictor_columns].copy()

# Primary outcome
y = model_data["aki_label"].astype(int).copy()

# Keep identifiers separately for patient-level splitting
identifiers = model_data[["subject_id", "stay_id", "hadm_id"]].copy()

# Secondary outcome in the same labelled population
y_stage2 = model_data["observed_stage2_positive"].astype(int).copy()

# Check the prepared tables
print("Predictor table shape:", X.shape)
print("Primary positives:", y.sum())
print("Secondary positives:", y_stage2.sum())
print("Unique patients:", identifiers["subject_id"].nunique())

print("\nPredictor columns:")
print(X.columns.tolist())

Predictor table shape: (113097, 13)
Primary positives: 7866
Secondary positives: 2254
Unique patients: 66090

Predictor columns:
['age', 'gender', 'arrival_transport', 'temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'acuity', 'pain_numeric', 'pain_status', 'chiefcomplaint']


Export data

In [50]:
from pathlib import Path
import shutil
from google.colab import files

# Create an export folder
export_folder = Path("/content/aki_handoff")
export_folder.mkdir(exist_ok=True)

# Prepare the modelling table
id_columns = ["subject_id", "stay_id", "hadm_id"]

modelling_export = model_data[
    id_columns + predictor_columns + ["aki_label"]
].copy()

modelling_export["aki_stage2_label"] = (
    model_data["observed_stage2_positive"].astype(int)
)

# Save the cleaned modelling data
modelling_export.to_csv(
    export_folder / "modelling_data.csv", index=False
)

# Save the full eligible cohort, including unresolved encounters
# Predictor values in this table are BEFORE the section 6 cleaning
cohort.to_csv(
    export_folder / "full_cohort_audit.csv", index=False
)

# Save original predictor values for the labelled encounters
raw_predictors = labelled[id_columns + predictors].copy()

raw_predictors.to_csv(
    export_folder / "labelled_predictors_before_cleaning.csv",
    index=False
)

# Save missingness and outcome counts
missing_summary.to_csv(export_folder / "predictor_missingness.csv")

cohort["label_state"].value_counts().rename_axis(
    "label_state"
).reset_index(name="encounters").to_csv(
    export_folder / "outcome_counts.csv", index=False
)

# Save the exact list of model inputs
pd.DataFrame({"predictor": predictor_columns}).to_csv(
    export_folder / "predictor_columns.csv", index=False
)

print("Modelling encounters exported:", len(modelling_export))
print("Full cohort encounters exported:", len(cohort))

Modelling encounters exported: 113097
Full cohort encounters exported: 195340
